In [51]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

MODEL_SAVE_PATH = '/content/drive/MyDrive/DR_Project/models'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(f'{MODEL_SAVE_PATH}/mobilenetv2_phase1_best.keras',
                     monitor='val_accuracy', save_best_only=True)
]

history_phase1 = model.fit(
    train_datagen.flow(X_train_img, y_train_cat, batch_size=32),
    validation_data=(X_val_img, y_val_cat),
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 53s 651ms/step - accuracy: 0.4772 - loss: 1.3474 - val_accuracy: 0.3752 - val_loss: 1.3958
Epoch 2/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 441ms/step - accuracy: 0.4982 - loss: 1.2968 - val_accuracy: 0.4936 - val_loss: 1.4056
Epoch 3/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 35s 437ms/step - accuracy: 0.5076 - loss: 1.2737 - val_accuracy: 0.4390 - val_loss: 1.3974
Epoch 4/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 34s 424ms/step - accuracy: 0.5076 - loss: 1.2892 - val_accuracy: 0.3698 - val_loss: 1.4126
Epoch 5/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 35s 426ms/step - accuracy: 0.5088 - loss: 1.2706 - val_accuracy: 0.4936 - val_loss: 1.3661
Epoch 6/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 440ms/step - accuracy: 0.5021 - loss: 1.2757 - val_accuracy: 0.4845 - val_loss: 1.3747
Epoch 7/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 34s 417ms/step - accuracy: 0.5115 - loss: 1.2639 - val_accuracy: 0.4936 - val_loss: 1.3379
Epoch 8/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 34s 418ms/step - accuracy: 0.5084 - loss: 1.2576 - val_accu

In [50]:
import tensorflow as tf
print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("Built with CUDA:", tf.test.is_built_with_cuda())

GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Built with CUDA: True


In [52]:
# Unfreeze the last ~30 layers of the backbone for fine-tuning
base_model.trainable = True

# Keep early layers frozen (they capture generic edges/textures — no need to retrain)
# Only fine-tune the deeper, more specialized layers
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a much lower learning rate — critical for fine-tuning stability
model.compile(
    optimizer=Adam(learning_rate=0.00001),  # 100x lower than Phase 1
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Trainable parameters now:", sum([tf.size(w).numpy() for w in model.trainable_weights]))

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(f'{MODEL_SAVE_PATH}/mobilenetv2_phase2_best.keras',
                     monitor='val_accuracy', save_best_only=True)
]

history_phase2 = model.fit(
    train_datagen.flow(X_train_img, y_train_cat, batch_size=32),
    validation_data=(X_val_img, y_val_cat),
    epochs=30,
    callbacks=callbacks_phase2,
    verbose=1
)

Trainable parameters now: 1887877
Epoch 1/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 79s 650ms/step - accuracy: 0.5174 - loss: 1.2740 - val_accuracy: 0.4936 - val_loss: 1.3353
Epoch 2/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 457ms/step - accuracy: 0.5127 - loss: 1.2737 - val_accuracy: 0.4936 - val_loss: 1.3319
Epoch 3/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 444ms/step - accuracy: 0.5181 - loss: 1.2600 - val_accuracy: 0.4936 - val_loss: 1.3286
Epoch 4/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 35s 424ms/step - accuracy: 0.5224 - loss: 1.2543 - val_accuracy: 0.4936 - val_loss: 1.3283
Epoch 5/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 34s 416ms/step - accuracy: 0.5205 - loss: 1.2445 - val_accuracy: 0.4936 - val_loss: 1.3293
Epoch 6/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 449ms/step - accuracy: 0.5279 - loss: 1.2317 - val_accuracy: 0.4882 - val_loss: 1.3301
Epoch 7/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 34s 421ms/step - accuracy: 0.5291 - loss: 1.2367 - val_accuracy: 0.4863 - val_loss: 1.3283
Epoch 8/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 439ms/step - accuracy

In [53]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Reload your saved [0,1] normalized images and convert to MobileNetV2's expected [-1,1] range
# First, undo the 0-1 normalization back to 0-255, then apply the correct preprocessing
X_train_img_fixed = preprocess_input(X_train_img * 255.0)
X_val_img_fixed = preprocess_input(X_val_img * 255.0)
X_test_img_fixed = preprocess_input(X_test_img * 255.0)

print("Fixed range check — min:", X_train_img_fixed.min(), "max:", X_train_img_fixed.max())
# Should print something close to min: -1.0, max: 1.0

Fixed range check — min: -1.0 max: 1.0


In [54]:
# Rebuild the base model fresh
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(5, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Rebuild the augmentation generator on the FIXED data
train_datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.15, horizontal_flip=True, brightness_range=[0.8, 1.2], fill_mode='nearest'
)
train_datagen.fit(X_train_img_fixed)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(f'{MODEL_SAVE_PATH}/mobilenetv2_phase1_best.keras', monitor='val_accuracy', save_best_only=True)
]

history_phase1 = model.fit(
    train_datagen.flow(X_train_img_fixed, y_train_cat, batch_size=32),
    validation_data=(X_val_img_fixed, y_val_cat),
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 65s 646ms/step - accuracy: 0.6262 - loss: 1.0317 - val_accuracy: 0.6321 - val_loss: 1.0127
Epoch 2/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 448ms/step - accuracy: 0.7066 - loss: 0.7974 - val_accuracy: 0.6648 - val_loss: 0.9420
Epoch 3/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 453ms/step - accuracy: 0.7187 - loss: 0.7430 - val_accuracy: 0.6776 - val_loss: 0.8649
Epoch 4/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 39s 432ms/step - accuracy: 0.7234 - loss: 0.7156 - val_accuracy: 0.6703 - val_loss: 0.9635
Epoch 5/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 452ms/step - accuracy: 0.7413 - loss: 0.6894 - val_accuracy: 0.6521 - val_loss: 0.9964
Epoch 6/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 452ms/step - accuracy: 0.7413 - loss: 0.6739 - val_accuracy: 0.7049 - val_loss: 0.8555
Epoch 7/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 39s 480ms/step - accuracy: 0.7437 - loss: 0.6549 - val_accuracy: 0.6302 - val_loss: 1.0843
Epoch 8/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 38s 475ms/step - accuracy: 0.7374 - loss: 0.6571 - val_accu

In [55]:
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Trainable parameters now:", sum([tf.size(w).numpy() for w in model.trainable_weights]))

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(f'{MODEL_SAVE_PATH}/mobilenetv2_phase2_best.keras',
                     monitor='val_accuracy', save_best_only=True)
]

history_phase2 = model.fit(
    train_datagen.flow(X_train_img_fixed, y_train_cat, batch_size=32),
    validation_data=(X_val_img_fixed, y_val_cat),
    epochs=30,
    callbacks=callbacks_phase2,
    verbose=1
)

Trainable parameters now: 1887877
Epoch 1/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 77s 691ms/step - accuracy: 0.5466 - loss: 1.4079 - val_accuracy: 0.7614 - val_loss: 0.6883
Epoch 2/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 452ms/step - accuracy: 0.7031 - loss: 0.7760 - val_accuracy: 0.7559 - val_loss: 0.7324
Epoch 3/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 459ms/step - accuracy: 0.7191 - loss: 0.7451 - val_accuracy: 0.7505 - val_loss: 0.7726
Epoch 4/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 37s 457ms/step - accuracy: 0.7343 - loss: 0.6961 - val_accuracy: 0.7413 - val_loss: 0.8253
Epoch 5/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 38s 471ms/step - accuracy: 0.7343 - loss: 0.6877 - val_accuracy: 0.7486 - val_loss: 0.8182
Epoch 6/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 439ms/step - accuracy: 0.7554 - loss: 0.6457 - val_accuracy: 0.7377 - val_loss: 0.8119
Epoch 7/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 38s 469ms/step - accuracy: 0.7585 - loss: 0.6508 - val_accuracy: 0.7359 - val_loss: 0.8052
Epoch 8/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 36s 441ms/step - accuracy

In [1]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras.utils import to_categorical

# Evaluate on the held-out test set (never touched during training)
test_loss, test_accuracy = model.evaluate(X_test_img_fixed, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Get predictions for detailed metrics
y_pred_proba = model.predict(X_test_img_fixed)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\n=== Classification Report ===")
print(classification_report(y_test_img, y_pred, target_names=['Stage 0', 'Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test_img, y_pred))

# AUC-ROC (multi-class, one-vs-rest)
auc = roc_auc_score(y_test_cat, y_pred_proba, multi_class='ovr')
print(f"\nAUC-ROC (macro, one-vs-rest): {auc:.4f}")


NameError: name 'model' is not defined

In [6]:
import os
print("Files in models folder:", os.listdir(MODEL_SAVE_PATH))

Files in models folder: ['clinical_model_xgboost.pkl', 'clinical_threshold.pkl', 'shap_summary.png', 'shap_waterfall_patient1.png', 'mobilenetv2_phase1_best.keras', 'mobilenetv2_phase2_best.keras', 'mobilenetv2_phase3_balanced.keras', 'mobilenetv2_final.keras', 'model2_final_results.json']


In [8]:
from tensorflow.keras.models import load_model

# Load the Phase 2 checkpoint (should be the epoch-1 best weights, ~76% val accuracy)
model = load_model(f'{MODEL_SAVE_PATH}/mobilenetv2_phase2_best.keras')
print("Model loaded successfully.")

Model loaded successfully.


In [9]:
test_loss, test_accuracy = model.evaluate(X_test_img_fixed, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

y_pred_proba = model.predict(X_test_img_fixed)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\n=== Classification Report ===")
print(classification_report(y_test_img, y_pred, target_names=['Stage 0', 'Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test_img, y_pred))

auc = roc_auc_score(y_test_cat, y_pred_proba, multi_class='ovr')
print(f"\nAUC-ROC (macro, one-vs-rest): {auc:.4f}")

Test Accuracy: 0.7636
Test Loss: 0.6822
18/18 ━━━━━━━━━━━━━━━━━━━━ 11s 317ms/step

=== Classification Report ===
              precision    recall  f1-score   support

     Stage 0       0.92      0.99      0.95       271
     Stage 1       1.00      0.07      0.13        56
     Stage 2       0.62      0.87      0.73       150
     Stage 3       0.45      0.17      0.25        29
     Stage 4       0.35      0.27      0.31        44

    accuracy                           0.76       550
   macro avg       0.67      0.48      0.47       550
weighted avg       0.78      0.76      0.72       550


=== Confusion Matrix ===
[[268   0   3   0   0]
 [ 11   4  40   0   1]
 [ 10   0 131   3   6]
 [  0   0   9   5  15]
 [  2   0  27   3  12]]

AUC-ROC (macro, one-vs-rest): 0.9009


In [10]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights based on training set imbalance
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_img),
    y=y_train_img
)
class_weight_dict = dict(enumerate(class_weights_arr))
print("Class weights:", class_weight_dict)

# Continue training the current model with class weights, low LR (still fine-tuning)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks_phase3 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ModelCheckpoint(f'{MODEL_SAVE_PATH}/mobilenetv2_phase3_balanced.keras',
                     monitor='val_accuracy', save_best_only=True)
]

history_phase3 = model.fit(
    X_train_img_fixed, y_train_cat,
    validation_data=(X_val_img_fixed, y_val_cat),
    batch_size=32,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase3,
    verbose=1
)

Class weights: {0: np.float64(0.4058590657165479), 1: np.float64(1.9791505791505792), 2: np.float64(0.7333333333333333), 3: np.float64(3.797037037037037), 4: np.float64(2.476328502415459)}
Epoch 1/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 53s 428ms/step - accuracy: 0.6937 - loss: 1.2523 - val_accuracy: 0.7705 - val_loss: 0.6839
Epoch 2/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.7183 - loss: 1.1272 - val_accuracy: 0.7723 - val_loss: 0.7015
Epoch 3/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.7366 - loss: 1.0385 - val_accuracy: 0.7814 - val_loss: 0.6979
Epoch 4/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.7472 - loss: 0.9838 - val_accuracy: 0.7851 - val_loss: 0.6829
Epoch 5/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.7476 - loss: 0.9568 - val_accuracy: 0.7942 - val_loss: 0.6721
Epoch 6/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.7550 - loss: 0.9152 - val_accuracy: 0.7851 - val_loss: 0.6708
Epoch 7/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/s

In [11]:
test_loss, test_accuracy = model.evaluate(X_test_img_fixed, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

y_pred_proba = model.predict(X_test_img_fixed)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\n=== Classification Report ===")
print(classification_report(y_test_img, y_pred, target_names=['Stage 0', 'Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test_img, y_pred))

auc = roc_auc_score(y_test_cat, y_pred_proba, multi_class='ovr')
print(f"\nAUC-ROC (macro, one-vs-rest): {auc:.4f}")

Test Accuracy: 0.7600
Test Loss: 0.6405
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step

=== Classification Report ===
              precision    recall  f1-score   support

     Stage 0       0.93      0.99      0.96       271
     Stage 1       0.48      0.52      0.50        56
     Stage 2       0.68      0.69      0.69       150
     Stage 3       0.26      0.21      0.23        29
     Stage 4       0.44      0.27      0.34        44

    accuracy                           0.76       550
   macro avg       0.56      0.54      0.54       550
weighted avg       0.74      0.76      0.75       550


=== Confusion Matrix ===
[[267   4   0   0   0]
 [  8  29  19   0   0]
 [  9  23 104  13   1]
 [  1   1   7   6  14]
 [  1   4  23   4  12]]

AUC-ROC (macro, one-vs-rest): 0.9109


In [12]:
model.save(f'{MODEL_SAVE_PATH}/mobilenetv2_final.keras')
print("Final image model saved.")

# Save evaluation results for your report
import json
final_results = {
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'auc_roc_macro': float(auc),
    'class_weights_used': {str(k): float(v) for k, v in class_weight_dict.items()}
}
with open(f'{MODEL_SAVE_PATH}/model2_final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Results saved.")

Final image model saved.
Results saved.
